# 计算机视觉

> 参考：动手学深度学习 v2，第 13 章

深度学习已深刻改变了计算机视觉领域，推动了医疗诊断、自动驾驶、安防监控等应用的发展。本笔记涵盖图像增广、迁移学习、目标检测、语义分割和风格迁移等核心主题，侧重理论原理。


---
### 1.图像增广

深度神经网络通常需要大量训练数据。当数据不足时，模型容易过拟合——记住了训练集的细节而不是学到泛化规律。**图像增广（Image Augmentation）** 通过对已有训练图像施加随机变换来人工扩充数据集，同时减少模型对某些特定属性（如目标的绝对位置、精确色调）的依赖，从而提升泛化能力。
> 关键直觉：一张猫的照片，无论水平翻转、裁剪还是调整亮度，它仍然是猫。利用这一先验知识可以显著扩充有效训练样本。

#### 常见增广方法:

**翻转（Flipping）**

| 类型 | 说明 | 使用场景 |
|------|------|----------|
| 水平翻转（左右） | 最常用，对绝大多数视觉任务安全 | 通用 |
| 垂直翻转（上下） | 使用较少，适用于卫星图等特定数据集 | 特定场景 |

**随机裁剪（Random Cropping）**

随机裁剪是最重要的增广方法之一。典型策略：

- 随机选取原图面积的 **10%–100%** 作为裁剪区域
- 宽高比在 **0.5–2.0** 之间随机变化
- 裁剪后缩放到统一输入尺寸

效果：迫使模型学习局部特征，对目标位置和尺度具有鲁棒性。

**颜色抖动（Color Jittering）**

随机调整图像的颜色属性：

- **亮度（Brightness）**：如调整到原亮度的 50%–150%
- **对比度（Contrast）**：拉伸或压缩像素值分布
- **饱和度（Saturation）**：调整颜色的鲜艳程度
- **色调（Hue）**：整体偏移颜色频谱

效果：使模型对光照、拍摄条件的变化更鲁棒。

### 使用策略

- **训练阶段**：随机应用多种增广（翻转 + 裁剪 + 颜色变换）
- **测试/推理阶段**：仅使用确定性变换（如中心裁剪 + 归一化），不加随机性

多种增广方法通过 `Compose` 串联使用，顺序施加。

In [ ]:
import torchvision.transforms as T

train_transform = T.Compose([
    T.RandomHorizontalFlip(),
    T.RandomResizedCrop(224, scale=(0.1, 1.0), ratio=(0.5, 2.0)),
    T.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

---
### 2. 微调（迁移学习）


在实际任务中，特定领域的标注数据往往远少于 ImageNet 这类大规模数据集（120 万张图像，1000 类）。从零训练一个大型网络极易过拟合。**微调（Fine-tuning）** 利用源数据集上学到的通用特征，以最小的数据和计算成本适配目标任务。
CNN 的浅层学到的是通用的低级特征（边缘、纹理、颜色块），深层学到的是更特定于任务的高级语义特征。当源域和目标域相关时，低层特征可以直接复用，只需微调高层。

#### 微调四步流程

```
Step 1: 在大规模源数据集（如 ImageNet）上预训练模型
           ↓
Step 2: 复制预训练模型的全部结构和权重（去掉输出层）
           ↓
Step 3: 替换输出层 — 新建输出维度 = 目标类别数 的全连接层，随机初始化
           ↓
Step 4: 在目标数据集上训练：输出层用大学习率，其余层用小学习率
```


In [ ]:
import torch
import torchvision.models as models

model = models.resnet18(pretrained=True)

# 替换输出层（目标类别数 = 2，如热狗/非热狗）
num_features = model.fc.in_features
model.fc = torch.nn.Linear(num_features, 2)

# 差异化学习率：输出层 10 倍学习率
optimizer = torch.optim.SGD([
    {'params': [p for n, p in model.named_parameters() if 'fc' not in n], 'lr': 1e-4},
    {'params': model.fc.parameters(), 'lr': 1e-3}
], momentum=0.9, weight_decay=1e-4)

---
### 3.目标检测与边界框


图像分类回答「图中有什么」，而**目标检测（Object Detection）**同时回答「图中有什么、在哪里」，需要输出每个目标的类别和空间位置。

典型应用包括自动驾驶（识别行人、车辆、交通标志）、安防监控、医疗影像分析。一般来说目标检测需要画框和识别。

#### 3.1 边界框（Bounding Box）

边界框是最常用的目标位置表示方式，通常为矩形框。存在两种等价的表示方法：

**方法一：角坐标表示法（Corner Format）**

$$\text{box} = (x_1, y_1, x_2, y_2)$$

其中 $(x_1, y_1)$ 是左上角坐标，$(x_2, y_2)$ 是右下角坐标。

**方法二：中心表示法（Center Format）**

$$\text{box} = (c_x, c_y, w, h)$$

其中 $(c_x, c_y)$ 是框中心坐标，$w$ 是宽度，$h$ 是高度。

两种表示的相互转换：

$$c_x = \frac{x_1 + x_2}{2}, \quad c_y = \frac{y_1 + y_2}{2}, \quad w = x_2 - x_1, \quad h = y_2 - y_1$$

#### 3.2 坐标系约定

图像坐标系以左上角为原点，x 轴向右，y 轴向下。坐标通常以像素为单位，也可归一化到 $[0, 1]$。

In [ ]:
import torch

def box_corner_to_center(boxes):
    x1, y1, x2, y2 = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    cx = (x1 + x2) / 2
    cy = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    return torch.stack([cx, cy, w, h], dim=-1)

def box_center_to_corner(boxes):
    cx, cy, w, h = boxes[:, 0], boxes[:, 1], boxes[:, 2], boxes[:, 3]
    x1 = cx - w / 2
    y1 = cy - h / 2
    x2 = cx + w / 2
    y2 = cy + h / 2
    return torch.stack([x1, y1, x2, y2], dim=-1)


#### 3.3 锚框

目标检测的核心挑战是：目标的位置、尺度和宽高比千变万化。**锚框（Anchor Box）** 提供了一种系统化的解决方案——预先生成覆盖各种位置、尺度和形状的候选框，再由网络预测每个锚框是否包含目标以及精确位置的修正量。

对于 $h \times w$ 的输入图像，以每个像素为中心，用不同的缩放比和宽高比组合生成锚框。

给定缩放比 $s \in (0, 1]$ 和宽高比 $r > 0$，锚框的宽和高为：

$$\text{width} = h \cdot s \cdot \sqrt{r}, \qquad \text{height} = \frac{h \cdot s}{\sqrt{r}}$$

**实践简化策略**：不枚举所有 $(s, r)$ 组合，只使用包含 $s_1$ 或 $r_1$ 的组合，共 $n + m - 1$ 个锚框（$n$ 个缩放比，$m$ 个宽高比）。整张图共生成 $wh(n+m-1)$ 个锚框。




**交并比**

交并比是衡量两个边界框重叠程度的标准指标：

$$\text{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

| IoU 值 | 含义 |
|---------|------|
| 0 | 两框完全不重叠 |
| 0.5 | 通常作为正样本的最低阈值 |
| 1 | 两框完全重合 |



训练时需为每个锚框分配标签（正/负样本）和真实偏移量。

**分配算法（最大 IoU 匹配 + 阈值过滤）**：

```
1. 计算所有锚框与所有真实框的 IoU 矩阵
2. 迭代地找最大 IoU 对，将真实框分配给对应锚框（确保每个真实框至少有一个匹配）
3. 对剩余未分配锚框，若与某真实框的 IoU > 阈值（如 0.5），分配给该真实框
4. 其余锚框标记为背景（负样本）
```

**偏移量编码**：对真实偏移量做归一化变换以提高数值稳定性：

$$\Delta x = \frac{x^* - x_a}{w_a}, \quad \Delta w = \log \frac{w^*}{w_a}$$

其中带 $^*$ 的是真实框属性，带 $a$ 的是锚框属性。

#### 预测时的非极大值抑制（NMS）

模型对每个锚框输出预测，同一目标可能有多个高置信度预测框。**NMS（Non-Maximum Suppression）** 用于去除冗余检测：

```
Algorithm NMS(boxes, scores, threshold e):
  按置信度降序排列所有预测框 -> 列表 L
  结果列表 R = []
  while L 不为空:
      取 L 中置信度最高的框 B，加入 R
      从 L 中移除所有与 B 的 IoU > e 的框
  return R
```

NMS 结束后，R 中任意两框的 IoU 都 $\leq \varepsilon$，有效消除重复检测。

---
### 4.多尺度目标检测


现实场景中，同一类目标的尺寸差异极大（如近处的大车和远处的小车）。在输入图像的每个像素处生成锚框的计算量也很大：对于 $561 \times 728$ 的图像，若每像素生成 5 个锚框，共需处理超过 200 万个锚框。


**核心思路**：不在原始图像上生成所有锚框，而是在 CNN **不同层的特征图**上分别生成锚框。

| 特征图层次 | 空间分辨率 | 感受野 | 适合检测 |
|-----------|-----------|--------|----------|
| 浅层（靠近输入） | 高（密集采样） | 小 | 小目标 |
| 深层（远离输入） | 低（稀疏采样） | 大 | 大目标 |


特征图上每个空间位置对应输入图像上一个固定大小的**感受野（Receptive Field）**。网络只需利用该感受野内的信息来预测以该位置为中心的锚框。更深层的特征图虽然分辨率低，但每个位置的感受野更大，可以感知更大范围的上下文来预测大目标。多尺度检测思想后来发展为 **FPN（Feature Pyramid Network）**：不仅从不同深度的特征图上生成锚框，还通过自顶向下的路径融合深层语义信息和浅层空间细节，构建多尺度特征金字塔，显著提升小目标检测性能。

---
### 5. 单发多框检测（SSD）

**SSD（Single Shot MultiBox Detector）** 是多尺度目标检测的经典实现，采用「基础网络 + 多尺度特征块」的一阶段检测框架，可在单次前向传播中完成检测。

```
输入图像
   |
基础网络（VGG / ResNet）  <- 提取初始特征
   |
特征块 1 -> 预测头（分类 + 定位）
   |（下采样）
特征块 2 -> 预测头（分类 + 定位）
   |（下采样）
   ...（共 5 个尺度）
   |
NMS -> 最终检测结果
```

**5.1 类别预测层**

在某个特征图上，每个空间位置生成 $a$ 个锚框，共 $q$ 个目标类别（加背景共 $q+1$ 类）。类别预测层使用 $3 \times 3$ 卷积核，输出通道数为 $a(q+1)$，保持特征图的空间维度。

**5.2 边界框预测层**

结构与类别预测层类似，每个锚框预测 4 个偏移量：

$$\text{输出通道数} = a \times 4$$

网络预测的是相对于锚框的**偏移量**（归一化中心偏移和对数宽高比），而非绝对坐标。

**5.3 多尺度特征块**

每个下采样块通常包含：
- 两个 $3 \times 3$ 卷积层
- 一个 $2 \times 2$ 最大池化层（步幅 2，将空间尺寸减半）

靠近输出的特征图较小但感受野较大，适合检测较少但较大的目标。

**5.4 损失函数**

SSD 的总损失为分类损失和回归损失的加权和：

$$\mathcal{L} = \mathcal{L}_{\text{cls}} + \alpha \cdot \mathcal{L}_{\text{loc}}$$

| 损失项 | 函数 | 说明 |
|--------|------|------|
| $\mathcal{L}_{\text{cls}}$ 分类损失 | 交叉熵 | 正样本 + 困难负样本参与 |
| $\mathcal{L}_{\text{loc}}$ 定位损失 | Smooth L1 | 仅正样本锚框参与，掩码屏蔽负样本 |

**难负样本挖掘（Hard Negative Mining）**：负样本数量远多于正样本，通常按负:正 = 3:1 选取损失最大的负样本，缓解类别不平衡问题。

---
### 6. R-CNN系列

R-CNN 系列是两阶段检测器的代表，经历了从低效到高效的经典演进历程。

**6.1 R-CNN（2014）——起点**

**核心思路**：先用传统方法（选择性搜索 Selective Search）生成约 2000 个候选区域，再对每个候选区域独立运行 CNN 提取特征，最后用 SVM 分类和线性回归定位。

```
图像 -> 选择性搜索 -> ~2000 个候选框 -> 逐一 CNN 提取特征 -> SVM 分类 + 回归定位
```

**致命缺陷**：每张图需要对约 2000 个候选区域分别进行前向传播，推理极慢（约 47 秒/张）。

**6.2 Fast R-CNN（2015）——共享计算**

**核心创新**：整张图只做**一次** CNN 前向传播，所有候选区域共享特征图。

关键技术——**RoI Pooling（兴趣区域池化）**：将特征图上任意形状的候选区域映射为固定大小的特征向量（如 $7 \times 7$），再接全连接层进行分类和定位。

```
图像 -> CNN（一次前向）-> 特征图
                              ^
选择性搜索 -> 候选框 -> RoI Pooling -> 分类 + 定位
```

性能提升：推理从 47 秒/张降至约 2 秒/张（但选择性搜索仍是瓶颈）。

**6.3 Faster R-CNN（2015）——端到端**

**核心突破**：用可学习的**区域提议网络（RPN, Region Proposal Network）**替代传统选择性搜索，实现完全端到端训练。

RPN 共享主干网络的特征图，在其上滑动窗口生成锚框，预测每个锚框是否为目标（二分类）及精确位置，再将高质量候选框送入检测头。

```
图像 -> 主干 CNN -> 特征图
                      |-> RPN -> 候选区域（可学习）
                      |-> RoI Pooling -> 分类 + 定位
```

推理速度：约 0.2 秒/张，精度显著提升。

**6.4 Mask R-CNN（2017）——实例分割**

**核心扩展**：在 Faster R-CNN 基础上增加像素级分割分支，同时实现目标检测 + 实例分割。

两项关键改进：

1. **RoI Align**：用双线性插值替换 RoI Pooling，消除量化误差，精确保留空间位置信息
2. **全卷积分割头**：在每个候选区域上预测一个二值掩码（每个类别各一个 $28 \times 28$ 掩码）

---
### 7. 语义分割


计算机视觉中的图像理解任务，按粒度由粗到细排列：

| 任务 | 粒度 | 输出 | 示例 |
|------|------|------|------|
| 图像分类 | 图像级 | 一个类别标签 | 这张图有一只猫 |
| 目标检测 | 目标级 | 边界框 + 类别 | 猫在 (x1,y1,x2,y2) |
| 语义分割 | 像素级（类别） | 每像素的类别标签 | 每个像素标为猫/背景 |
| 实例分割 | 像素级（实例） | 每像素的实例标签 | 区分第1只猫和第2只猫的像素 |

**语义分割**将图像中的每个像素分配到一个语义类别，但不区分同一类别的不同个体。

**实例分割**在语义分割的基础上进一步区分同类不同个体。


---
### 8. 转置卷积


语义分割等像素级任务需要输出与输入同分辨率的特征图。CNN 的池化和步幅卷积会逐步压缩空间尺寸，因此需要**上采样（Upsampling）**来恢复分辨率。

| 方法 | 是否可学习 | 特点 |
|------|-----------|------|
| 最近邻插值 | 否 | 计算快，边缘锯齿明显 |
| 双线性插值 | 否 | 平滑，无法适应数据 |
| **转置卷积** | **是** | 权重从数据中学习，更灵活 |



普通卷积：通过滑动窗口对输入进行**加权求和**，输出尺寸 <= 输入尺寸。

转置卷积：对输入的**每个元素**与卷积核相乘，得到局部输出，将所有局部输出叠加，输出尺寸 >= 输入尺寸。


卷积运算可以写成矩阵乘法形式：$\mathbf{y} = \mathbf{W} \mathbf{x}$

转置卷积对应的前向计算使用 $\mathbf{W}$ 的转置：$\mathbf{y} = \mathbf{W}^\top \mathbf{x}$，这也是名字的由来。

注意：转置卷积**不是**卷积的逆运算（值不完全还原），但可以还原**尺寸**。

In [ ]:
import torch
import torch.nn as nn

conv = nn.Conv2d(1, 1, kernel_size=2, stride=2)          # 下采样
trans_conv = nn.ConvTranspose2d(1, 1, kernel_size=2, stride=2)  # 上采样

x = torch.randn(1, 1, 4, 4)
y = conv(x)
z = trans_conv(y)
print(f'输入: {x.shape}  -> 卷积后: {y.shape}  -> 转置卷积后: {z.shape}')

---
### 9. 全卷积网络（FCN）



传统 CNN 分类器末尾的全连接层将空间信息「压平」为固定长度向量，无法输出空间预测图。**FCN（Fully Convolutional Network）** 将全连接层替换为卷积层，使网络可以接受任意尺寸输入并输出与之对应的像素级预测图。


```
输入图像（H x W x 3）
      |
编码器（Encoder）
  预训练 CNN 主干（如 ResNet-18）
  空间尺寸逐步缩小到 H/32 x W/32
      |
1x1 卷积（将通道数转为类别数，如 Pascal VOC 的 21 类）
      |
解码器（Decoder）
  转置卷积 x5（步幅 2，逐步恢复分辨率，共上采样 32 倍）
      |
输出（H x W x 21）：每个像素位置上的类别预测分数
```

**双线性插值**

为加速收敛，FCN 使用**双线性插值**初始化转置卷积层的权重（而非随机初始化）。双线性插值用最近的 4 个输入像素加权平均计算输出值，提供良好的训练起点。

**跳跃连接（Skip Connection）**

原始 FCN 论文中引入了跳跃连接，将浅层高分辨率特征与深层语义特征相加融合（FCN-16s、FCN-8s）：

```
深层语义特征（低分辨率）
    ^（上采样 2x）
    + 中间层特征（高分辨率）  <- 跳跃连接
    ^（上采样 16x）
    -> 输出
```

跳跃连接的作用：补偿深层特征丢失的空间细节，改善分割边界精度（特别是小目标）。这一思想被后续网络大量采用（U-Net、DeepLab 等）。

---
### 10. 神经风格迁移


**神经风格迁移（Neural Style Transfer）** 将一幅图像的**艺术风格**（笔触、色彩、纹理）迁移到另一幅图像的**内容**上，合成一幅兼具两者特征的新图像。

```
内容图像（一张风景照）
         +
风格图像（梵高《星夜》）
         |
合成图像（用梵高风格画的风景）
```

**10.1 核心思想**

利用预训练 CNN（如 VGG-19）提取特征：
- **内容特征**：深层特征图，捕捉目标的结构和全局语义
- **风格特征**：各层特征图之间的**统计关系**（通道相关性），捕捉纹理和笔触

优化对象不是网络权重，而是**合成图像本身**（从内容图像初始化）。

**10.2 特征层的选择（VGG-19）**

| 用途 | 选取的层 | 原因 |
|------|---------|------|
| 内容层 | 第 4 卷积块的最后一层（深层） | 保留全局语义，避免过度保留细节纹理 |
| 风格层 | 每个卷积块的第一个卷积层（共 5 层） | 捕捉从低级纹理到高级风格的多层次特征 |

**10.3 三种损失函数**

**内容损失（Content Loss）**：衡量合成图像与内容图像在内容特征上的差异，使用均方误差：

$$\mathcal{L}_{\text{content}} = \frac{1}{2} \sum_{l} \left\| F^l_{\text{合成}} - F^l_{\text{内容}} \right\|_F^2$$

**风格损失（Style Loss）**：风格用 **Gram 矩阵**来度量，它捕捉了特征图中通道之间的相关性（即纹理的统计特征）。

给定第 $l$ 层特征图，形状为 $C \times H \times W$，将其重塑为 $C \times (HW)$ 的矩阵 $\mathbf{F}$，则：

$$\mathbf{G}^l = \frac{1}{C \cdot HW} \mathbf{F} \mathbf{F}^\top \quad \in \mathbb{R}^{C \times C}$$

$G^l_{ij}$ 衡量第 $i$ 通道和第 $j$ 通道特征的相关程度，即「哪些纹理倾向于同时出现」。

风格损失：

$$\mathcal{L}_{\text{style}} = \sum_{l} \left\| \mathbf{G}^l_{\text{合成}} - \mathbf{G}^l_{\text{风格}} \right\|_F^2$$

**全变差损失（Total Variation Loss）**：对合成图像施加平滑约束，减少噪声和高频伪影

$$\mathcal{L}_{\text{TV}} = \sum_{i,j} \left( |x_{i,j} - x_{i+1,j}| + |x_{i,j} - x_{i,j+1}| \right)$$

**10.4 总损失与训练流程**

$$\mathcal{L}_{\text{total}} = \lambda_{c} \cdot \mathcal{L}_{\text{content}} + \lambda_{s} \cdot \mathcal{L}_{\text{style}} + \lambda_{\text{TV}} \cdot \mathcal{L}_{\text{TV}}$$

三个权重超参数控制内容保真度和风格强度之间的平衡。

**训练流程**：

```
初始化：合成图像 X <- 内容图像
固定 VGG-19 权重（不更新网络，只更新图像像素）

循环（约 500 轮）：
  1. 前向传播：提取 X、内容图像、风格图像的多层特征
  2. 计算三种损失并加权求和
  3. 反向传播：计算 dL/dX
  4. Adam 更新 X（仅更新图像像素值）
  5. 每 50 轮，学习率衰减 x 0.8
```

In [ ]:
import torch

def gram_matrix(feature_map):
    B, C, H, W = feature_map.shape
    F = feature_map.view(B, C, H * W)
    return torch.bmm(F, F.transpose(1, 2)) / (C * H * W)

def content_loss(y_hat, y):
    return torch.mean((y_hat - y.detach()) ** 2)

def style_loss(y_hat, y):
    return torch.mean((gram_matrix(y_hat) - gram_matrix(y).detach()) ** 2)

def tv_loss(img):
    return (torch.abs(img[:, :, 1:, :] - img[:, :, :-1, :]).mean() +
            torch.abs(img[:, :, :, 1:] - img[:, :, :, :-1]).mean())

---
## 总结

| 章节 | 核心方法 | 关键思想 |
|------|---------|----------|
| 图像增广 | 翻转、裁剪、颜色抖动 | 数据层面的正则化，减少过拟合 |
| 微调 | 迁移学习 + 差异化学习率 | 复用预训练特征，适配小数据集 |
| 目标检测 | 边界框表示（角坐标/中心） | 在分类基础上增加空间定位 |
| 锚框 | IoU、NMS | 系统化候选框 + 去除冗余检测 |
| 多尺度检测 | 特征金字塔 FPN | 浅层检测小目标，深层检测大目标 |
| SSD | 一阶段检测 | 多尺度特征图 + 联合损失 |
| R-CNN 系列 | 两阶段检测 | RPN + RoI Pooling，端到端演进 |
| 语义分割 | 像素级分类 | 全卷积网络，输出与输入同分辨率 |
| 转置卷积 | 可学习上采样 | 卷积权重转置，还原特征图尺寸 |
| FCN | 编码器-解码器 + 跳跃连接 | 深层语义 + 浅层细节融合 |
| 风格迁移 | Gram 矩阵 + 三种损失 | 优化图像像素，分离内容与风格 |